In [0]:
import os, sys
sys.path.append(os.path.abspath("../src"))
import mlflow
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from churn.dbx import get_config
from churn.config import SEGMENT_FEATURES, TRAIN_SNAPSHOTS, in_window
from churn import modeling as M

cfg = get_config(dbutils)
mlflow.set_registry_uri("databricks-uc")                     # le registry = Unity Catalog
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/attrition-banque-segmentation")

feats = spark.table(cfg.t("client_features")).toPandas()
train = feats[in_window(feats.snapshot_date, TRAIN_SNAPSHOTS)]
print(len(train), "lignes d'entraînement")

In [0]:
with mlflow.start_run(run_name="kmeans_segmentation") as run:
    best_k, scores = M.choose_k(train)                       # teste k = 3, 4, 5, 6
    for k, s in scores.items():
        mlflow.log_metric("silhouette", s, step=k)           # une courbe silhouette selon k
    model = M.SegmentationModel(best_k).fit(train)

    mlflow.log_params({"k": best_k, "features": ",".join(SEGMENT_FEATURES)})
    mlflow.log_metric("best_silhouette", model.silhouette_)
    mlflow.log_dict({str(k): v for k, v in model.names.items()}, "segment_names.json")

    sample = train[SEGMENT_FEATURES].head(100)
    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=model,
        code_paths=["../src/churn"],                          # embarque notre package dans le modèle
        signature=infer_signature(sample, model.predict(None, sample)),
        input_example=sample.head(5),
        registered_model_name=cfg.segment_model_name)         # → workspace.attrition_dev.client_segmentation

MlflowClient().set_registered_model_alias(cfg.segment_model_name, "champion", info.registered_model_version)
print(f"k = {best_k} | segments = {model.names} | version = {info.registered_model_version}")

In [0]:
train = train.assign(segment_comportemental=model.predict(None, train))
display(train.groupby("segment_comportemental")[SEGMENT_FEATURES].mean().round(1)
        .assign(nb_lignes=train.segment_comportemental.value_counts()))